# Results

Reads `results/tables/*.json` -- the per-seed numbers behind the manuscript's
tables -- and recomputes the statistics with `left.evaluation.metrics`. No GPU,
no datasets.


In [1]:
import json
import numpy as np

from left.evaluation import compare_arms, mean_sd

mixer = json.load(open("../results/tables/mixer_5seed_raw.json"))
print(mixer["_note"])
print("\narms:", sorted({k.split("|")[1] for k in mixer["common"]}))

Depth-2 MLP-Mixer, width 76, token/channel-mixing hidden 38/304; one shared activation module in every token- and channel-mixing MLP. Common: lr 1e-3, wd 0.05. Tuned: 15-cell grid (30 ep, seed 0), best cell x 5 seeds x 100 ep.

arms: ['elu', 'fact_fixed', 'gelu', 'hardswish', 'leaky_relu', 'mish', 'relu', 'selu', 'silu', 'softplus', 'tanh']


## MLP-Mixer, common configuration

The attention-free target: the curve was learned inside a self-attention
network and is installed in an architecture that has none.

In [2]:
DATASETS = ["fmnist", "cifar10", "cifar100", "food101"]
SEEDS = [0, 1, 2, 3, 4]

def table(block):
    """The stored accuracies are percentages; left.evaluation works in
    fractions (and reports margins in pp), so divide by 100 on the way in."""
    out = {}
    for key, accs in block.items():
        ds, act = key.split("|")
        out.setdefault(ds, {})[act] = {s: a / 100.0 for s, a in zip(SEEDS, accs)}
    return out

common = table(mixer["common"])
acts = ["fact_fixed", "relu", "leaky_relu", "gelu", "silu", "mish",
        "hardswish", "softplus", "elu", "selu", "tanh"]

header = f"{'activation':<12}" + "".join(f"{d:>16}" for d in DATASETS)
print(header); print("-" * len(header))
for a in acts:
    row = f"{a:<12}"
    for d in DATASETS:
        if a in common.get(d, {}):
            m, s = mean_sd(common[d][a].values())
            row += f"{100*m:>11.2f}+-{100*s:<4.2f}"
        else:
            row += f"{'-':>16}"
    print(row)

activation            fmnist         cifar10        cifar100         food101
----------------------------------------------------------------------------
fact_fixed        91.42+-0.18      75.99+-0.16      47.49+-0.38      35.09+-0.22
relu              90.80+-0.22      75.47+-0.35      47.08+-0.46      34.92+-0.35
leaky_relu        90.64+-0.39      75.51+-0.13      47.20+-0.48      34.94+-0.40
gelu              89.84+-0.13      75.32+-0.48      45.99+-0.60      36.15+-0.47
silu              88.90+-0.29      73.75+-0.30      45.07+-0.83      35.15+-0.30
mish              88.94+-0.25      74.03+-0.30      45.17+-0.58      35.25+-0.25
hardswish         87.81+-0.41      72.78+-0.42      44.05+-0.78      34.39+-0.26
softplus          86.98+-0.42      68.26+-1.06      35.33+-1.84      32.70+-0.12
elu               86.52+-0.22      70.19+-0.56      41.27+-0.78      30.82+-0.38
selu              86.02+-0.22      64.97+-0.22      36.07+-0.46      24.90+-0.25
tanh              86.53+-0.18      6

## Is the margin real?

Paired across seeds, Holm-corrected over the four datasets. `fact_fixed` is the
transferred curve.

In [3]:
rows = compare_arms({d: common[d] for d in DATASETS}, "fact_fixed", "gelu")
for d in DATASETS:
    r = rows[d]
    print(f"{d:<10} {r['margin_pp']:+6.2f}pp  t={r['t']:+6.2f}  "
          f"p={r['p_raw']:.4f}  p_holm={r['p_holm']:.4f}  "
          f"{'significant' if r['significant'] else 'ns'}")

fmnist      +1.58pp  t=+13.31  p=0.0002  p_holm=0.0007  significant
cifar10     +0.67pp  t= +3.74  p=0.0201  p_holm=0.0348  significant
cifar100    +1.50pp  t= +4.31  p=0.0126  p_holm=0.0348  significant
food101     -1.05pp  t= -4.41  p=0.0116  p_holm=0.0348  significant


Food-101 on the Mixer is the one cell of the paper's 16 where the
transferred curve does not come first under the common configuration -- GELU
wins it. Tuning lr and wd per activation flips it back; the appendix reports
both conditions.

## The other three architectures

In [4]:
raw = json.load(open("../results/tables/transfer_5seed_raw.json"))
print("seeds:", raw["seeds"])
print("keys:", list(raw["acc"])[:6], "...")
print("params:", raw["params"])

seeds: [0, 1, 2, 3, 4]
keys: ['vit|fmnist|fact_fixed', 'vit|fmnist|relu', 'vit|fmnist|leaky_relu', 'vit|fmnist|gelu', 'vit|fmnist|silu', 'vit|fmnist|mish'] ...
params: {'vit|fmnist': 105098, 'vit|cifar10': 108106, 'vit|cifar100': 113956, 'vit|food101': 123237, 'swin|fmnist': 102354, 'swin|cifar10': 104402, 'swin|cifar100': 110252, 'swin|food101': 110317, 'linvit|fmnist': 105098, 'linvit|cifar10': 108106, 'linvit|cifar100': 113956, 'linvit|food101': 123237}


In [5]:
# Margin of the transferred curve over GELU, per architecture x dataset,
# straight from the stored per-seed accuracies.
stats = json.load(open("../results/tables/transfer_5seed_stats.json"))
print(json.dumps(stats, indent=2)[:1500])

{
  "vit|fmnist": {
    "means": {
      "relu": 90.404,
      "leaky_relu": 90.26599999999999,
      "gelu": 89.09,
      "silu": 87.96199999999999,
      "mish": 87.91000000000001,
      "hardswish": 87.534,
      "softplus": 86.27400000000002,
      "elu": 86.32000000000001,
      "selu": 86.24199999999999,
      "tanh": 86.60999999999999,
      "fact_fixed": 90.93799999999999
    },
    "sds": {
      "relu": 0.45500549447232175,
      "leaky_relu": 0.2669831455354447,
      "gelu": 0.2402082429892867,
      "silu": 0.5661006977561495,
      "mish": 0.3451811118818608,
      "hardswish": 0.5808442131931765,
      "softplus": 0.3746731909277787,
      "elu": 0.09433981132056572,
      "selu": 0.5111946791585368,
      "tanh": 0.2045727254547855,
      "fact_fixed": 0.10377861051295587
    },
    "strongest": "relu",
    "cmp": {
      "relu": {
        "diff": 0.5339999999999975,
        "sd": 0.4102194534636335,
        "wins": 5
      },
      "leaky_relu": {
        "diff": 0.671